In [5]:
import os
import sys
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import pickle
import seaborn as sns

# Add the parent directory to the path so we can import your custom modules
scripts_path = os.path.abspath('../scripts/deep_learning_4')
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

scripts_root = os.path.abspath('../scripts')
if scripts_root not in sys.path:
    sys.path.append(scripts_root)
    
from online_selector import OnlineModeSelector
from ml_config import MLConfig

# Recreate the windowing function for the notebook
def create_sliding_windows_for_episode(X, y, window_size=MLConfig.WINDOW_SIZE):
    X_win, y_win = [], []
    for t in range(X.shape[0] - window_size + 1):
        X_win.append(X[t:t+window_size, :])
        y_win.append(y[t+window_size-1, :])
    return np.array(X_win), np.array(y_win)

# 1. Load Data (We will just use the test set)
X_test_d2d = np.load("../data/model_ready/d2d/X_test.npy")
y_test_d2d = np.load("../data/model_ready/d2d/y_test.npy")
X_test_cell = np.load("../data/model_ready/cellular/X_test.npy")
y_test_cell = np.load("../data/model_ready/cellular/y_test.npy")

# Select a single random episode (e.g., Episode 0) to visualize
ep_idx = 0 
X_ep_d2d, y_ep_d2d = create_sliding_windows_for_episode(X_test_d2d[ep_idx], y_test_d2d[ep_idx])
X_ep_cell, y_ep_cell = create_sliding_windows_for_episode(X_test_cell[ep_idx], y_test_cell[ep_idx])

# 2. Load the CNN Models and Error Params
model_name = 'cnn'
model_d2d = tf.keras.models.load_model(f"../models/d2d/{model_name}/{model_name}_model.keras")
model_cell = tf.keras.models.load_model(f"../models/cellular/{model_name}/{model_name}_model.keras")

with open(f"../models/d2d/{model_name}/{model_name}_error_params_kde.pkl", "rb") as f:
    err_params_d2d = pickle.load(f)

# 3. Generate Predictions for the whole episode
preds_d2d = model_d2d.predict(X_ep_d2d, verbose=0).flatten()
preds_cell = model_cell.predict(X_ep_cell, verbose=0).flatten()

# 4. Simulate the Online Selector step-by-step
selector = OnlineModeSelector(model_name=model_name)
current_mode = 'D2D'

# Arrays to store the data for plotting
time_steps = []
true_sinr_log = []
pred_sinr_log = []
upper_bound_log = []
lower_bound_log = []
throughput_log = []
switch_points_x = []
switch_points_y = []

print("Simulating 1 Episode...")
for t in range(len(preds_d2d)):
    # Get physical ground truth
    true_d2d = y_ep_d2d[t, 0]
    true_cell = y_ep_cell[t, 0]
    
    # Make decision
    new_mode, logs = selector.make_decision(preds_d2d[t], preds_cell[t], current_mode)
    
    # Track switches
    if new_mode != current_mode:
        switch_points_x.append(t)
        # We will append the Y-value (throughput) after we calculate it below
        
    current_mode = new_mode
    
    # Calculate actual realized throughput based on the physical signal of the chosen mode
    if current_mode == 'D2D':
        actual_tput = selector.ts.shannon_throughput(true_d2d)
        active_true_sinr = true_d2d
        active_pred_sinr = preds_d2d[t]
        margin_upper = err_params_d2d['upper_bound']
        margin_lower = err_params_d2d['lower_bound']
    else:
        actual_tput = selector.ts.shannon_throughput(true_cell)
        active_true_sinr = true_cell
        active_pred_sinr = preds_cell[t]
        # Assuming D2D params for simplicity in graph, or you can load cell params
        margin_upper = err_params_d2d['upper_bound'] 
        margin_lower = err_params_d2d['lower_bound']
        
    if t in switch_points_x and len(switch_points_y) < len(switch_points_x):
        switch_points_y.append(actual_tput)

    # Log data
    time_steps.append(t)
    true_sinr_log.append(active_true_sinr)
    pred_sinr_log.append(active_pred_sinr)
    upper_bound_log.append(active_pred_sinr + margin_upper)
    lower_bound_log.append(active_pred_sinr + margin_lower)
    throughput_log.append(actual_tput)

print("Simulation Complete. Ready to plot!")

Initializing Online Mode Selector (CNN model | AR constraint)


FileNotFoundError: [Errno 2] No such file or directory: 'models/d2d/cnn/cnn_error_params_kde.pkl'

In [2]:
# Set plain background like the paper
sns.set_style("ticks")
plt.figure(figsize=(16, 8))

# Plot the 3 lines
plt.plot(time_steps, true_sinr_log, color='blue', linewidth=2, label='Real', alpha=0.8)
plt.plot(time_steps, pred_sinr_log, color='red', linewidth=2, label='Predicted', alpha=0.8)
plt.plot(time_steps, upper_bound_log, color='orange', linewidth=3, label='Upper and lower bounds')
plt.plot(time_steps, lower_bound_log, color='orange', linewidth=3)

# Formatting to match the research paper style
plt.xlim(0, len(time_steps))
plt.xlabel('Time Interval', fontsize=16, fontweight='bold')
plt.ylabel('SINR', fontsize=16, fontweight='bold')

# Remove top and right borders (spines) to match the uploaded graph
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.legend(loc='upper right', fontsize=12)

# Save and show
os.makedirs('../data/results/system_viz', exist_ok=True)
plt.savefig('../data/results/system_viz/sinr_confidence_intervals.png', dpi=300, bbox_inches='tight')
plt.show()

NameError: name 'time_steps' is not defined

<Figure size 1600x800 with 0 Axes>

In [ ]:
plt.figure(figsize=(16, 8))

# Plot the throughput line
plt.plot(time_steps, throughput_log, color='#1f77b4', linewidth=2)

# Scatter plot the red dots where a mode switch occurred
plt.scatter(switch_points_x, switch_points_y, color='red', s=80, zorder=5, label='Mode Switch')

# Formatting to match the research paper style
plt.xlim(0, len(time_steps))
plt.ylim(0, max(throughput_log) * 1.1)
plt.xlabel('Time Interval', fontsize=14)
plt.ylabel('Throughput (Mbps)', fontsize=14)

plt.legend(loc='lower right', fontsize=14, frameon=True, shadow=True)

# Save and show
plt.savefig('../data/results/system_viz/throughput_mode_switches.png', dpi=300, bbox_inches='tight')
plt.show()